In [5]:
# Core libraries
import pandas as pd
import numpy as np
import re
from datetime import datetime
import sys
print(sys.executable)
# The star of the show
from google_play_scraper import app, reviews, Sort

print("Libraries loaded successfully!")

c:\Users\cbe\Desktop\Web_Scraping\fintech-review-analytics\venv\Scripts\python.exe
Libraries loaded successfully!


In [21]:
from google_play_scraper import app, reviews, Sort
import pandas as pd

# ==============================
# App IDs (fixed + clarified)
# ==============================
APPS = {
    "CBE Mobile Banking": "com.combanketh.mobilebanking",
    "Dashen Bank": "com.dashen.dashensuperapp",
    "BOA Mobile Banking": "com.boa.boaMobileBanking"
}

TARGET_REVIEWS = 500
all_reviews = []

# ==============================
# STEP 1: App Metadata + Reviews
# ==============================
for bank_name, app_id in APPS.items():

    print("\n" + "=" * 50)
    print(f"Processing: {bank_name}")
    print("=" * 50)

    # Get app metadata
    info = app(app_id, lang='en', country='et')

    print(f"App Title   : {info['title']}")
    print(f"Score       : {info['score']}")
    print(f"Ratings     : {info['ratings']:,}")
    print(f"Reviews     : {info['reviews']:,}")
    print(f"Installs    : {info['installs']}")

    # ==============================
    # STEP 2: Scrape Reviews (400+)
    # ==============================
    collected = []
    token = None

    while len(collected) < TARGET_REVIEWS:
        batch, token = reviews(
            app_id,
            lang='en',
            country='et',
            sort=Sort.NEWEST,
            count=500,
            continuation_token=token
        )

        if not batch:
            break

        collected.extend(batch)

        if token is None:
            break

    print(f"Collected Reviews: {len(collected)}")

    # ==============================
    # STEP 3: Format Data
    # ==============================
    for r in collected[:TARGET_REVIEWS]:

        # skip empty reviews
        if not r['content'] or len(r['content'].strip()) < 3:
            continue

        all_reviews.append({
            "bank": bank_name,
            "app_name": info['title'],
            "review_text": r['content'],
            "rating": r['score'],
            "review_date": r['at'],
            "source": "Google Play"
        })

# ==============================
# STEP 4: Create Dataset
# ==============================
df = pd.DataFrame(all_reviews)

print("\nFINAL DATASET SHAPE:", df.shape)
print(df.head())

# Save dataset
df.to_csv("data/bank_reviews.csv", index=False)
print("\nSaved to data/bank_reviews.csv")


Processing: CBE Mobile Banking
App Title   : Commercial Bank of Ethiopia
Score       : 4.2877455
Ratings     : 48,329
Reviews     : 9,307
Installs    : 5,000,000+
Collected Reviews: 500

Processing: Dashen Bank
App Title   : Dashen Bank
Score       : 4.22913
Ratings     : 5,618
Reviews     : 1,023
Installs    : 1,000,000+
Collected Reviews: 500

Processing: BOA Mobile Banking
App Title   : BoA Mobile
Score       : 4.3922596
Ratings     : 9,219
Reviews     : 1,461
Installs    : 1,000,000+
Collected Reviews: 500

FINAL DATASET SHAPE: (1458, 6)
                 bank                     app_name  \
0  CBE Mobile Banking  Commercial Bank of Ethiopia   
1  CBE Mobile Banking  Commercial Bank of Ethiopia   
2  CBE Mobile Banking  Commercial Bank of Ethiopia   
3  CBE Mobile Banking  Commercial Bank of Ethiopia   
4  CBE Mobile Banking  Commercial Bank of Ethiopia   

                     review_text  rating         review_date       source  
0                     incredible       5 2026-05-1

In [22]:
import pandas as pd

df = pd.read_csv("data/bank_reviews.csv")

print(df.shape)
df.head()

(1458, 6)


,bank,app_name,review_text,rating,review_date,source
0,CBE Mobile Banking,Commercial Bank of Ethiopia,incredible,5,2026-05-14 10:53:56,Google Play
1,CBE Mobile Banking,Commercial Bank of Ethiopia,best app for financial sector,5,2026-05-14 05:44:01,Google Play
2,CBE Mobile Banking,Commercial Bank of Ethiopia,it's a good application,5,2026-05-13 20:28:58,Google Play
3,CBE Mobile Banking,Commercial Bank of Ethiopia,thank you cbe,5,2026-05-13 17:16:37,Google Play
4,CBE Mobile Banking,Commercial Bank of Ethiopia,is good,5,2026-05-13 16:18:45,Google Play


In [26]:
import pandas as pd

# Load your existing data
df = pd.read_csv("data/bank_reviews.csv")

# -------------------------------
# 1. Rename columns
# -------------------------------
df = df.rename(columns={
    "review_text": "review",
    "review_date": "date"
})

# -------------------------------
# 2. Normalize date format
# -------------------------------
df["date"] = pd.to_datetime(df["date"], errors="coerce")

# Convert to YYYY-MM-DD
df["date"] = df["date"].dt.strftime("%Y-%m-%d")

# -------------------------------
# 3. Keep only required columns
# -------------------------------
df = df[["app_name", "bank","review", "rating", "date", "source",]]

# -------------------------------
# 4. Drop missing values (important)
# -------------------------------
df = df.dropna(subset=["review", "rating", "date"])

# -------------------------------
# 5. Optional: remove duplicates
# -------------------------------
df = df.drop_duplicates()

# -------------------------------
# 6. Save cleaned dataset
# -------------------------------
df.to_csv("data/cleaned_bank_reviews.csv", index=False)

print("Final shape:", df.shape)
print(df.head())

Final shape: (1439, 6)
                      app_name                bank  \
0  Commercial Bank of Ethiopia  CBE Mobile Banking   
1  Commercial Bank of Ethiopia  CBE Mobile Banking   
2  Commercial Bank of Ethiopia  CBE Mobile Banking   
3  Commercial Bank of Ethiopia  CBE Mobile Banking   
4  Commercial Bank of Ethiopia  CBE Mobile Banking   

                          review  rating        date       source  
0                     incredible       5  2026-05-14  Google Play  
1  best app for financial sector       5  2026-05-14  Google Play  
2        it's a good application       5  2026-05-13  Google Play  
3                  thank you cbe       5  2026-05-13  Google Play  
4                        is good       5  2026-05-13  Google Play  
